# Asian Options: Averaging Over Time

**Notebook 7** of a series implementing **Stamatopoulos et al., *"Option Pricing using Quantum Computers"*** [arXiv:1905.02666](https://arxiv.org/abs/1905.02666), Section 4.2.2.

A companion write-up is available on [Medium Blog](https://medium.com/@arpitha.rajeev37/implementing-option-pricing-using-quantum-computers-in-qiskit-part-5-asian-and-barrier-options-dc079e82fb06).

Notebook 6 combined two *different* assets into one basket, using custom weights (60% Apple, 40% Google). An Asian option combines the *same* asset, sampled at several time checkpoints, using *equal* weights -- the payoff depends on the average price across those checkpoints, not just the final one.

Structurally, this notebook is Notebook 6 with different input: same weighted-sum operator (Notebook 5), same comparator-plus-payoff-rotation pipeline (Notebooks 3-4), just pointed at time checkpoints of one asset instead of two different assets.

**What this notebook does:**
1. Define a 2-checkpoint Asian call and compute the classical ground truth.
2. Load both checkpoints' price distributions.
3. Reuse the weighted-sum operator with equal weights to compute the average price.
4. Reuse the comparator-plus-rotation pipeline from Notebook 6.
5. Verify, run AE, and execute on real hardware.

## Setup

In [1]:
import numpy as np
import sympy as sp
from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation, WeightedAdder, IntegerComparator
from qiskit.quantum_info import Statevector
from qiskit_algorithms import EstimationProblem, IterativeAmplitudeEstimation
from qiskit.primitives import StatevectorSampler

## The contract

An Asian call option, averaged over 2 checkpoints of the same asset. `Average = (S_t1 + S_t2) / 2`, payoff `max(0, Average - K)`.

Reusing price levels similar to earlier notebooks, but now as two checkpoints of one asset rather than two different assets -- the same 4 price levels can plausibly apply to the same stock at two different points in time, just with different probability distributions at each checkpoint (a later checkpoint typically has a wider, more uncertain spread).

In [2]:
t1_prices = [30, 40, 50, 60]
t1_probs = [0.20, 0.35, 0.30, 0.15]
t2_prices = [30, 40, 50, 60]
t2_probs = [0.25, 0.30, 0.25, 0.20]

K = 42

expected_payoff_truth = 0.0
for p1, pr1 in zip(t1_prices, t1_probs):
    for p2, pr2 in zip(t2_prices, t2_probs):
        avg = (p1 + p2) / 2
        expected_payoff_truth += pr1 * pr2 * max(0, avg - K)

print('Ground truth expected payoff:', expected_payoff_truth)

Ground truth expected payoff: 4.092499999999999


## Component 1: distribution loading for both checkpoints

Same pattern as Notebook 6's two-asset loading -- 16 joint amplitudes, one per (t1 price, t2 price) combination.

In [3]:
amplitudes = np.zeros(16)
for i2, (p2, pr2) in enumerate(zip(t2_prices, t2_probs)):
    for i1, (p1, pr1) in enumerate(zip(t1_prices, t1_probs)):
        index = (i2 << 2) | i1
        amplitudes[index] = np.sqrt(pr1 * pr2)

state_prep = StatePreparation(amplitudes)

sv_check = Statevector(state_prep)
result_check = sv_check.probabilities_dict()
for label in ['0000', '0101', '1011']:
    q3, q2, q1, q0 = int(label[0]), int(label[1]), int(label[2]), int(label[3])
    t2_idx = q3*2 + q2
    t1_idx = q1*2 + q0
    expected = t1_probs[t1_idx] * t2_probs[t2_idx]
    got = result_check.get(label, 0)
    print(f'{label}: t1=${t1_prices[t1_idx]}, t2=${t2_prices[t2_idx]}, got P={got:.4f}, expected P={expected:.4f}')

0000: t1=$30, t2=$30, got P=0.0500, expected P=0.0500
0101: t1=$40, t2=$40, got P=0.1050, expected P=0.1050
1011: t1=$60, t2=$50, got P=0.0375, expected P=0.0375


## Component 2a: the weighted sum -- equal weights, computing an average

For a 2-checkpoint average, each checkpoint carries weight `1/2`. Deriving the per-qubit integer weights follows exactly Notebook 5/6's symbolic-expansion method -- only the weight values themselves differ (`1/2, 1/2` instead of `0.6, 0.4`).

In [4]:
i, j = sp.symbols('i j')
w_t1, w_t2 = sp.Rational(1, 2), sp.Rational(1, 2)
t1_intercept, t1_slope = 30, 10
t2_intercept, t2_slope = 30, 10

S_avg = w_t1*(t1_intercept + t1_slope*i) + w_t2*(t2_intercept + t2_slope*j)
S_avg_expanded = sp.expand(S_avg)

baseline = float(S_avg_expanded.subs({i: 0, j: 0}))
i_coeff = S_avg_expanded.coeff(i)
j_coeff = S_avg_expanded.coeff(j)

weights = [int(i_coeff*1), int(i_coeff*2), int(j_coeff*1), int(j_coeff*2)]
print('baseline:', baseline, '| weights:', weights)

adder = WeightedAdder(num_state_qubits=4, weights=weights)
print(adder.qregs)

baseline: 30.0 | weights: [5, 10, 5, 10]
[QuantumRegister(4, 'state'), QuantumRegister(5, 'sum'), QuantumRegister(4, 'carry'), QuantumRegister(1, 'control')]


### Verifying the adder reconstructs the true average

In [5]:
t1_i, t2_j = 2, 1  # t1=$50, t2=$40
true_avg = float(w_t1)*t1_prices[t1_i] + float(w_t2)*t2_prices[t2_j]

q1_a, q0_a = t1_i // 2, t1_i % 2
q1_b, q0_b = t2_j // 2, t2_j % 2

qc_check = QuantumCircuit(adder.num_qubits)
if q0_a: qc_check.x(0)
if q1_a: qc_check.x(1)
if q0_b: qc_check.x(2)
if q1_b: qc_check.x(3)
qc_check.append(adder, range(adder.num_qubits))

sv_c = Statevector(qc_check)
result_c = max(sv_c.probabilities_dict(), key=sv_c.probabilities_dict().get)

sizes = [r.size for r in adder.qregs]
names = [r.name for r in adder.qregs]
reversed_bits = result_c[::-1]
idx = 0
decoded = {}
for name, size in zip(names, sizes):
    decoded[name] = reversed_bits[idx:idx+size][::-1]
    idx += size
sum_val = int(decoded['sum'], 2)
reconstructed = baseline + sum_val

print('True average:  ', true_avg)
print('Reconstructed: ', reconstructed)
print('Match:', np.isclose(true_avg, reconstructed))

True average:   45.0
Reconstructed:  45.0
Match: True


## Component 2b: comparator and payoff rotation

Structurally identical to Notebook 6 -- comparator on the `sum` register against a shifted strike, then a payoff rotation using the same linear-approximation formula (Eq. 20-21).

In [6]:
K_shifted = K - baseline
sum_size = adder.qregs[1].size
print('K_shifted:', K_shifted, '| sum register width:', sum_size)

comparator = IntegerComparator(sum_size, K_shifted, geq=True)
print(comparator.qregs)

K_shifted: 12.0 | sum register width: 5
[QuantumRegister(5, 'state'), QuantumRegister(1, 'compare'), QuantumRegister(4, 'a0')]


In [7]:
max_price = max(t1_prices[-1], t2_prices[-1])
f_max = max(0, max_price - K)
c = 0.25

sum_max = 2**sum_size - 1
g_slope = 2*c / (sum_max - K_shifted) if sum_max != K_shifted else 0
rot_base = 2 * (np.pi/4 - c)
rot_strike_offset = -2 * g_slope * K_shifted
rot_angles = [2 * g_slope * (2**k) for k in range(sum_size)]

print('f_max:', f_max, '| rot_base:', rot_base, '| rot_angles:', rot_angles)

f_max: 18 | rot_base: 1.0707963267948966 | rot_angles: [0.05263157894736842, 0.10526315789473684, 0.21052631578947367, 0.42105263157894735, 0.8421052631578947]


## Assembling the full A operator

In [8]:
total_qubits = 4 + (adder.num_qubits - 4) + (comparator.num_qubits - sum_size) + 1
qc_A = QuantumCircuit(total_qubits)

qc_A.compose(state_prep, qubits=range(4), inplace=True)
qc_A.compose(adder, qubits=range(adder.num_qubits), inplace=True)

sum_qubits = list(range(4, 4 + sum_size))
comparator_extra = list(range(adder.num_qubits, adder.num_qubits + (comparator.num_qubits - sum_size)))
qc_A.compose(comparator, qubits=sum_qubits + comparator_extra, inplace=True)

comparator_qubit = comparator_extra[0]
payoff_qubit = total_qubits - 1

qc_A.ry(rot_base, payoff_qubit)
qc_A.cry(rot_strike_offset, comparator_qubit, payoff_qubit)
for k, angle in enumerate(rot_angles):
    qc_A.mcry(angle, [comparator_qubit, sum_qubits[k]], payoff_qubit)

print('Total qubits:', qc_A.num_qubits, '| payoff qubit index:', payoff_qubit)

Total qubits: 20 | payoff qubit index: 19


### Verifying against ground truth

In [9]:
sv_A = Statevector(qc_A)
result_A = sv_A.probabilities_dict()

raw_p1 = sum(v for k, v in result_A.items() if k[0] == '1')

def post_process(a):
    return (a - 0.5 + c) * (f_max - 0) / (2*c) + 0

estimated_payoff = post_process(raw_p1)
print('P(payoff=1):', raw_p1)
print('Reconstructed expected payoff:', estimated_payoff)
print('Ground truth:                ', expected_payoff_truth)

P(payoff=1): 0.3626764916230905
Reconstructed expected payoff: 4.056353698431258
Ground truth:                 4.092499999999999


## Amplitude Estimation

In [10]:
problem = EstimationProblem(
    state_preparation=qc_A,
    objective_qubits=[payoff_qubit],
    post_processing=post_process,
)

sampler = StatevectorSampler()
iae = IterativeAmplitudeEstimation(epsilon_target=0.1, alpha=0.05, sampler=sampler)
result_iae = iae.estimate(problem)

print('Ground truth: ', expected_payoff_truth)
print('IAE estimate: ', result_iae.estimation_processed)

Ground truth:  4.092499999999999
IAE estimate:  4.273476235554705


## Map → Optimize → Execute → Post-process on real hardware

Reusing the `HardwareTranspiledSampler` wrapper from Notebook 3/6, which dynamically transpiles the internally-generated circuits and patches the `shots` metadata key.

In [17]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSampler
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print('Using backend:', backend.name)

Using backend: ibm_marrakesh


In [18]:
pm_hw = generate_preset_pass_manager(optimization_level=3, backend=backend)

class HardwareTranspiledSampler:
    def __init__(self, sampler, pass_manager):
        self._sampler = sampler
        self._pm = pass_manager

    def run(self, pubs, **kwargs):
        transpiled_pubs = []
        for pub in pubs:
            if isinstance(pub, tuple):
                circuit = self._pm.run(pub[0])
                transpiled_pubs.append((circuit, *pub[1:]))
            else:
                circuit = self._pm.run(pub)
                transpiled_pubs.append(circuit)
        job = self._sampler.run(transpiled_pubs, **kwargs)
        original_result = job.result
        def patched_result(*args, **kwargs_res):
            res = original_result(*args, **kwargs_res)
            for pub_res in res:
                if 'shots' not in pub_res.metadata:
                    reg_name = next(iter(pub_res.data.keys()))
                    bit_array = getattr(pub_res.data, reg_name)
                    pub_res.metadata['shots'] = bit_array.num_shots
            return res
        job.result = patched_result
        return job

base_sampler = RuntimeSampler(mode=backend)
hardware_sampler = HardwareTranspiledSampler(base_sampler, pm_hw)

In [19]:
problem_hw = EstimationProblem(
    state_preparation=qc_A,
    objective_qubits=[payoff_qubit],
    post_processing=post_process,
)

iae_hw = IterativeAmplitudeEstimation(epsilon_target=0.2, alpha=0.05, sampler=hardware_sampler)
result_iae_hw = iae_hw.estimate(problem_hw)

print('Ground truth:                  ', expected_payoff_truth)
print('IAE estimate (real hardware): ', result_iae_hw.estimation_processed)

Ground truth:                   4.092499999999999
IAE estimate (real hardware):  4.785078173202812


## Results Summary

- The Asian option's average-price computation reuses Notebook 5's weighted-sum operator exactly, with equal weights instead of custom ones -- confirming the primitive generalizes across both use cases without modification.
- The comparator and payoff rotation stages are structurally identical to Notebook 6's basket option.
- Amplitude Estimation and the real-hardware execution pattern carry over unchanged from Notebooks 3, 6.
## What's Next

**Notebook 8** covers barrier options -- the one remaining new piece of circuit wiring in this series: multiple comparators (one per monitored checkpoint), combined with a logical **OR**, feeding into the payoff rotation as an additional condition alongside the usual strike comparator.